# Факторы деградации жилого фонда в РФ: предсказание аварийности многоквартирных домов методами машинного обучения
Исследовательский вопрос: Насколько точно алгоритмы машинного обучения способны предсказывать официальный статус аварийности многоквартирного дома, и каков относительный вклад институциональных (тип управления, регион) и инфраструктурных (возраст, площадь, материал) предикторов в эту вероятность?


## Данные
Датасет взят с сайта [Если быть точным](https://tochno.st/datasets/gisgkh).

**Что в данных**: износ и аварийность жилых домов

Данные об исправности и аварийности многоквартирных жилых домов  (МКД) России, внесенных в базу данных Государственной информационной системы жилищно-коммунального хозяйства (ГИС ЖКХ). В текущей версии набора данных сведения актуальны по состоянию на ноябрь 2025 года.

1. Институциональные и пространственные предикторы:
- **region_name** и **municipality_name** (Регион и Муниципалитет) — отражают уровень бюджетной обеспеченности и пространственное неравенство.
- **house_manegement_type** (Способ управления) — ключевая переменная, отражающая способность собственников к преодолению проблемы коллективного действия (ТСЖ, частная УК, непосредственное управление).
- **is_dormitory** (Статус общежития) — сконструированный бинарный признак (на основе атрибутов regional_property и municipal_property). Выделяет объекты с максимальным уровнем отчуждения проживающих от управления общим имуществом.
2. Инфраструктурные и архитектурные предикторы:
- **building_year** (Год постройки) и **plan_series** (Серия проекта здания) — базовые метрики объективного физического старения материалов и особенностей индустриального домостроения разных эпох.
- **total_area** и **residental_premises_area** (Общая и жилая площадь) — отражают масштаб объекта.
- **is_reconstructed** (Факт проведения реконструкции) — бинарный признак, созданный на основе наличия записи в поле reconstruction_year, указывающий на капитальные вложения в инфраструктуру здания.
- **internal_walls_type** и **house_type** — инженерно-конструктивные характеристики.
3. Целевая переменная (Target):
- На основе атрибута condition (Состояние здания) сформирована бинарная целевая переменная target. Домам со статусом «Аварийный» и «Ветхий» присвоен класс 1 (проблемный фонд), домам со статусом «Исправный» — класс 0.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data_gisgkh_157_v20260917.parquet")
df.head()

,full_address,region_name,region_oktmo,region_okato,municipality_oktmo,municipality_okato,postal_code,latitude,longitude,qc_geo,...,deterioration_estimation_date,condition,internal_walls_type,reconstruction_year,energy_efficency_class,house_management_type,management_organization_name,management_organization_link,regional_property,municipal_property
0,"обл Оренбургская, г Орск, ул Докучаева, д. 23",Оренбургская область,53000000,53000000,53723000001,53423000000,462401,51.229248,58.529400,Точные координаты,...,01.01.2017,Исправный,Стены из слоистых железобетонных панелей,<NA>,<NA>,УО,"ООО ""УК ""ДЕЙСТВИЕ""",https://dom.gosuslugi.ru/#!/organizationView/8...,<NA>,<NA>
1,"Респ Башкортостан, г Нефтекамск, ул Карцева, д...",Республика Башкортостан,80000000,80000000,80727000001,80427000000,452687,56.077099,54.225304,Точные координаты,...,26.03.2025,Исправный,Стены кирпичные,<NA>,<NA>,УО,"ООО ""Мегаполис""",https://dom.gosuslugi.ru/#!/organizationView/3...,False,False
2,"Респ Башкортостан, г Нефтекамск, ул Карцева, д...",Республика Башкортостан,80000000,80000000,80727000001,80427000000,452687,56.077385,54.224518,Точные координаты,...,26.03.2025,Исправный,Стены кирпичные,<NA>,<NA>,УО,"ООО ""Мегаполис""",https://dom.gosuslugi.ru/#!/organizationView/3...,False,False
3,"Респ Башкортостан, г Нефтекамск, ул Карцева, д...",Республика Башкортостан,80000000,80000000,80727000001,80427000000,452687,56.077663,54.223854,Точные координаты,...,03.04.2025,Исправный,Стены кирпичные,<NA>,С (Нормальный),УО,"ООО ""УК ""СВОЙ ДОМ""",https://dom.gosuslugi.ru/#!/organizationView/9...,False,False
4,"Респ Башкортостан, г Нефтекамск, ул Карцева, д...",Республика Башкортостан,80000000,80000000,80727000001,80427000000,452687,56.077927,54.223225,Точные координаты,...,26.03.2025,Исправный,Стены кирпичные,<NA>,С (Нормальный),УО,"ООО ""УК ""СВОЙ ДОМ""",https://dom.gosuslugi.ru/#!/organizationView/9...,False,False


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 956147 entries, 0 to 956146
Data columns (total 33 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   full_address                   956147 non-null  string 
 1   region_name                    956147 non-null  string 
 2   region_oktmo                   956147 non-null  string 
 3   region_okato                   956147 non-null  string 
 4   municipality_oktmo             938351 non-null  string 
 5   municipality_okato             938351 non-null  string 
 6   postal_code                    955464 non-null  string 
 7   latitude                       937904 non-null  float32
 8   longitude                      937904 non-null  float32
 9   qc_geo                         938351 non-null  string 
 10  timezone                       938351 non-null  string 
 11  flats                          938347 non-null  Int64  
 12  fias_id                        956147 non

In [8]:
TARGET = "condition"
RANDOM_STATE = 42

COLORS = {
    "accent": "#4654f5",
    "header": "#959ef2",
    "background": "#EEEEEE",
    "text": "#000000",
}

PLOT_COLOR = COLORS["accent"]

def style_table(df, caption, colors=COLORS, hide_index=True):
    styler = df.style.set_caption(caption)
    if hide_index:
        styler = styler.hide(axis="index")
    return styler.set_table_styles([
        # {"selector": "", "props": "border-collapse:collapse;"},
        {"selector": "caption", "props": f"font-size:16px; font-weight:bold; color:{colors['text']};"},
        {"selector": "th", "props": f"background-color:{colors['header']}; color:{colors['text']}; border:1px solid #ccc;"},
        {"selector": "td", "props": f"background-color:{colors['background']}; color:{colors['text']}; border:1px solid #ccc;"},
    ])

In [11]:
# Data overview

overview = pd.DataFrame({
    "Metric": ["Shape", "Number of features", "Missing Values", "Duplicate Rows"],
    "Value": [df.shape, df.drop(TARGET, axis=1).columns.size, df.isnull().sum().sum(), df.duplicated().sum()]
})
style_table(overview, "Dataset Overview")

Metric,Value
Shape,"(956147, 33)"
Number of features,32
Missing Values,4715831
Duplicate Rows,0


In [13]:
# Target distribution

target_distribution = df[TARGET].value_counts()
target_distribution
target_distribution_df = pd.DataFrame({
    "Condition": target_distribution.index,
    "Count": target_distribution.values,
    "Percentage": (target_distribution.values / target_distribution.sum()) * 100  
})
style_table(target_distribution_df, "Target distribution").format({
    "Count": "{:,.0f}",
    "Percentage": "{:.2f}%"
})

Condition,Count,Percentage
Исправный,"857,093",90.66%
Аварийный,"79,031",8.36%
Ветхий,"9,238",0.98%
Не выбран,29,0.00%


In [16]:
df.describe(include=["str"])

,full_address,region_name,region_oktmo,region_okato,municipality_oktmo,municipality_okato,postal_code,qc_geo,timezone,fias_id,...,plan_series,total_area,residental_premises_area,deterioration_estimation_date,condition,internal_walls_type,energy_efficency_class,house_management_type,management_organization_name,management_organization_link
count,956147,956147,956147,956147,938351,938351,955464,938351,938351,956147,...,720607,955943,836738,532152,945391,788145,133062,941722,658676,658972
unique,956006,85,85,85,20824,20892,22140,6,11,956021,...,57092,143186,194997,22825,4,2267,18,7,42706,55194
top,"край Алтайский, р-н Краснощёковский, с Краснощ...",Московская область,46000000,46000000,36701000001,60401000000,443099,Точные координаты,UTC+3,fca0a054-4f63-4dbf-b985-c87c47c5b4f9,...,Кирпичный,0,0,01.01.2018,Исправный,Стены кирпичные,Нет,УО,"ООО ""ГУК - КРАСНОДАР""",https://dom.gosuslugi.ru/#!/organizationView/4...
freq,2,52104,52104,52104,10098,7370,962,726044,550028,2,...,43184,1496,1884,8433,857093,353304,50239,629227,2727,2727


In [ ]:
# df[TARGET] = df[TARGET].map(lambda x: 1 if x in ['Аварийный', 'Ветхий'] else 0)

In [ ]:
df = df[(df['building_year'] > 1800) & (df['building_year'] <= 2025)]

df['is_reconstructed'] = df['reconstruction_year'].notna().astype(int)

df['is_dormitory'] = (df['regional_property'] == True) | (df['municipal_property'] == True)
df['is_dormitory'] = df['is_dormitory'].astype(int)

## Разведочный анализ данных (EDA)
*Проблема пропусков*: В исходных данных зафиксировано значительное число пропусков в категориальных переменных (например, plan_series, municipality_name). В логике нашего исследования отсутствие записи в государственном реестре само по себе является информативным сигналом (дефицит административного контроля). Поэтому пропуски в категориальных номинальных шкалах заполнены значением «Не указано». Пропуски в непрерывных числовых переменных (площадь) заполнены медианными значениями.

In [ ]:
cat_features =[
    'region_name', 'municipality_name', 'internal_walls_type',
    'house_manegement_type', 'plan_series', 'house_type'
]
df[cat_features] = df[cat_features].fillna('Не указано')

num_features =['building_year', 'total_area', 'residental_premises_area']
df[num_features] = df[num_features].fillna(df[num_features].median())

**График №1**: Дисбаланс целевой переменной
Этот график показывает, насколько мало в выборке аварийных домов по сравнению с нормальными.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))

ax = sns.countplot(data=df, x='Target')

plt.title('Распределение целевой переменной (Дисбаланс классов)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Статус дома', fontsize=12)
plt.ylabel('Количество многоквартирных домов', fontsize=12)

plt.xticks(ticks=[0, 1], labels=['Исправный (0)', 'Аварийный / Ветхий (1)'], fontsize=11)

for p in ax.patches:
    ax.annotate(format(p.get_height(), '.0f'),
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha = 'center', va = 'center',
                xytext = (0, 9),
                textcoords = 'offset points',
                fontsize=11)

plt.show()

Вывод по графику №1 (Дисбаланс классов):
Визуальный анализ распределения целевой переменной Target демонстрирует экстремальный дисбаланс (Class Imbalance). Объекты, официально признанные аварийными/ветхими, составляют абсолютное меньшинство генеральной совокупности. С точки зрения машинного обучения это означает риск того, что алгоритм будет склонен предсказывать мажоритарный класс (исправные дома). Для решения этой проблемы на этапе обучения модели потребуется применение механизма балансировки весов (class weights).

**График №2**: Возраст инфраструктуры

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(data=df, x='building_year', hue='Target',
             kde=True, bins=50, stat='density', common_norm=False,
             palette={0: 'blue', 1: 'red'}, alpha=0.4)

plt.title('Исторические волны застройки и плотность аварийного фонда', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Год постройки', fontsize=12)
plt.ylabel('Плотность распределения', fontsize=12)

plt.legend(title='Статус:', labels=['Аварийный / Ветхий (1)', 'Исправный (0)'])

plt.xlim(1850, 2025)

plt.show()

Вывод по графику №2 (Возраст инфраструктуры):
Гистограмма распределения жилищного фонда по годам постройки с разделением на классы аварийности подтверждает, что пик ветхого жилья смещен в сторону зданий, возведенных в первой половине XX века.

**График №3**: Институциональный срез



In [ ]:
plt.figure(figsize=(10, 6))

df_plot = df.copy()
df_plot['Target_percent'] = df_plot['Target'] * 100

ax = sns.barplot(data=df_plot, x='house_manegement_type', y='Target_percent',
                 errorbar=None)

plt.title('Доля проблемного фонда в зависимости от института управления', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Cпособ управления', fontsize=12)
plt.ylabel('Доля аварийных/ветхих домов (%)', fontsize=12)

plt.xticks(rotation=15, ha='right', fontsize=11)

for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha = 'center', va = 'center',
                xytext = (0, 9),
                textcoords = 'offset points',
                fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

Вывод по графику №3: Гистограмма иллюстрирует долю деградировавшего жилья в разрезе юридических форм управления. Отчетливо видна, что в сегменте домов, находящихся под управлением Товариществ собственников жилья (ТСЖ/ЖСК), доля аварийности крайне мала.

## Дизайн статистического исследования
В качестве базовой модели выбран алгоритм градиентного бустинга над деревьями решений — **CatBoost** (Categorical Boosting).

Его преимущества для данного исследования:
1. **Нативная обработка категорий**: CatBoost математически корректно работает со строковыми (текстовыми) признаками "под капотом", используя целевое кодирование (Target Encoding). Это позволяет модели находить сложные нелинейные связи между субъектом РФ, серией здания и его износом.
2. **Устойчивость к выбросам и неполным данным**: Древовидная архитектура бустинга робастна к нетипичным значениям (например, ошибкам ввода площади), которые неизбежно присутствуют в массивах государственной статистики.
3. **Встроенная компенсация дисбаланса**: Использование гиперпараметра auto_class_weights='Balanced' заставляет функцию потерь штрафовать модель пропорционально редкости аварийного класса, максимизируя способность алгоритма "отлавливать" проблемные объекты.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from catboost import CatBoostClassifier, Pool

features = cat_features + num_features + ['is_reconstructed', 'is_dormitory']
X = df[features]
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = CatBoostClassifier(
    iterations=500,
    cat_features=cat_features,
    auto_class_weights='Balanced', # Решает проблему редких аварийных домов
    random_seed=42,
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_test, y_test))

## Метрики оценки качества
Учитывая дисбаланс классов, стандартная метрика **Accuracy** (доля правильных ответов) является нерепрезентативной. Оценка качества алгоритма будет производиться на основе метрик **Precision** (точность), **Recall** (полнота), а также метрики **F1-score** и матрицы ошибок (**Confusion Matrix**). Для интерпретации социологических результатов будет применен анализ важности признаков (**Feature Importances**).

In [ ]:
preds = model.predict(X_test)
print(classification_report(y_test, preds))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

feature_importances = model.get_feature_importance()
sns.barplot(x=feature_importances, y=X.columns)
plt.title("Важность признаков (Feature Importances)")
plt.show()

### Вывод по Feature Importances:
1. Физические признаки
- Абсолютным лидером по силе предсказания ожидаемо стал **building_year** (Год постройки). Это подтверждает адекватность модели: законы физики и естественный износ материалов выступают первичным фактором риска. В связке с ним работают структурные характеристики: **total_area** (Общая площадь) и **internal_walls_type** (Материал стен). Высокая значимость площади объясняется тем, что основу аварийного фонда исторически составляют малоэтажные дома небольшой площади (например, двухэтажные деревянные бараки), в то время как крупные массивы поздней советской застройки деградируют медленнее.
2. Социально-пространственное неравенство
- Наибольший интерес представляет то, что пространственно-географические признаки — **municipality_name** (Муниципалитет) и **region_name** (Регион) — заняли 2-е и 3-е места среди всех факторов (не считая площади), в совокупности формируя колоссальный вес в принятии решения моделью.
Это доказывает выдвинутую гипотезу: статус «аварийного дома» — это социально-бюрократический конструкт. При одинаковом возрасте, площади и материале стен вероятность признания дома аварийным кардинально различается в зависимости от субъекта РФ и конкретного города. Это прямое отражение бюджетной обеспеченности: богатые муниципалитеты имеют средства на расселение и быстрее "навешивают" статус аварийности на старый фонд, тогда как в депрессивных регионах процесс бюрократически тормозится.
3. Роль институтов управления
- Институциональный признак **house_manegement_type** (Способ управления) вошел в ядро значимых факторов. Хотя его глобальный вес уступает возрасту и географии, его влияние остается статистически существенным. Алгоритм учитывает юридическую форму управления (например, защитный эффект ТСЖ по сравнению с муниципальными УК) при оценке риска деградации, что подтверждает важность прав коллективной собственности.
4. Специфика бинарных признаков
- Сконструированные бинарные признаки **is_reconstructed** (Факт реконструкции) и **is_dormitory** (Статус общежития) показали низкую глобальную важность. С методологической точки зрения машинного обучения это легко объяснимо: доля общежитий и реконструированных домов в генеральной совокупности (950 тыс. зданий) ничтожно мала. Алгоритм редко использует эти признаки для разбиения всей выборки. Однако их низкая глобальная важность не отменяет их высокой локальной значимости: для конкретного здания сам факт наличия статуса "общежитие" может выступать критическим триггером разрушения, что требует дальнейшего изучения

## Вывод
Таким образом, первоначальная исследовательская гипотеза подтвердилась: статус аварийности дома не является случайной, а алгоритмически выводим из набора объективных характеристик здания и институциональной среды.